<a href="https://colab.research.google.com/github/Priyansh08463/ML-Project/blob/main/email_spam_detect.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Email Spam Detection Machine Learning Model
This notebook demonstrates a complete end-to-end machine learning pipeline for detecting spam emails using TF-IDF and Naive Bayes classification.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.naive_bayes import MultinomialNB

## Upload and Load Your Kaggle Dataset
If you are using Google Colab, you can upload your downloaded Kaggle dataset (e.g., `spam.csv`) to the runtime files. Below is the code to load it, inspect its columns, and clean it up for our machine learning pipeline.

In [3]:
import os

file_path = '/content/drive/MyDrive/ColabNotebooks/emails.csv'

if os.path.exists(file_path):
    try:
        # Trying to load the dataset using common encodings
        df_kaggle = pd.read_csv(file_path, encoding='latin-1')
        print("Dataset loaded successfully from Google Drive!")
        print("Original Columns found:", df_kaggle.columns.tolist())
    except Exception as e:
        print(f"Error loading file: {e}")
else:
    print(f"The file was not found at {file_path}. Please double check the path in your Google Drive.")

Dataset loaded successfully from Google Drive!
Original Columns found: ['text', 'spam']


In [4]:
# Clean and standardize columns to ['text', 'label'] format
if 'text' in df_kaggle.columns and 'spam' in df_kaggle.columns:
    # Map 1 to 'spam' and 0 to 'ham'
    df_kaggle['label'] = df_kaggle['spam'].map({1: 'spam', 0: 'ham'})
    # Keep only the target columns
    df_kaggle = df_kaggle[['text', 'label']].copy()
    print("Kaggle dataset standardized!")
    display(df_kaggle.head())
    print("\nClass Distribution:")
    print(df_kaggle['label'].value_counts())
else:
    print("Standardization skipped. Please check your dataset's column names.")

Kaggle dataset standardized!


,text,label
0,Subject: naturally irresistible your corporate...,spam
1,Subject: the stock trading gunslinger fanny i...,spam
2,Subject: unbelievable new homes made easy im ...,spam
3,Subject: 4 color printing special request add...,spam
4,"Subject: do not have money , get software cds ...",spam



Class Distribution:
label
ham     4360
spam    1368
Name: count, dtype: int64


### Inspecting and Preprocessing the Kaggle Dataset
Many common Kaggle spam datasets (like the SMS Spam Collection) contain extra unnamed columns and have columns labeled `v1` (label) and `v2` (text). Let's clean and map them.

## 2. Train-Test Split
We split our data into a training set and a testing set.

In [5]:
# We will prioritize using your uploaded Kaggle dataset if it has been loaded and standardized
if not df_kaggle.empty and 'text' in df_kaggle.columns and 'label' in df_kaggle.columns:
    X = df_kaggle['text']
    y = df_kaggle['label']
    print("Using Kaggle dataset!")
else:
    X = df_kaggle['text']
    y = df_kaggle['label']
    print("Using temporary placeholder dataset!")

# Splitting the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")

Using Kaggle dataset!
Training set size: 4582
Testing set size: 1146


## 3. Feature Extraction & Model Training
We use `TfidfVectorizer` to convert text into numeric vectors, then train a Naive Bayes classifier.

In [6]:
# Initialize vectorizer
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)

# Fit and transform the training data, then transform the test data
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Initialize and train the Naive Bayes classifier
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

MultinomialNB()

## 4. Model Evaluation
Evaluate the performance of our model on the test dataset.

In [7]:
y_pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Model Accuracy: 97.82%

Classification Report:
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       856
        spam       0.99      0.93      0.96       290

    accuracy                           0.98      1146
   macro avg       0.98      0.96      0.97      1146
weighted avg       0.98      0.98      0.98      1146



## 5. Inference System
Let's write a function to test the spam classifier on custom, unseen email messages.

In [8]:
def predict_email(text):
    vectorized_text = vectorizer.transform([text])
    prediction = model.predict(vectorized_text)[0]
    probability = model.predict_proba(vectorized_text)[0]
    spam_prob = probability[1] if len(probability) > 1 else probability[0]
    return f"Prediction: {prediction.upper()}"

# Test examples
print(predict_email("Imagine if your city became your classroom Canva World Tour is coming to you, India! We’re inviting the world to imagine what’s possible and giving you the tools to bring it to life through free online learning and local workshops. Join an event and together we will learn new skills, swap design tips, and share creative inspiration"))
print(predict_email("Regarding your application to the live project in Full Stack Development with AI at Finlatics"))
print(predict_email("CONGRATULATIONS! You won a brand new car! Click this link to claim your reward now!"))

Prediction: SPAM
Prediction: HAM
Prediction: SPAM
